# Object Detection on Kaggle with pinned Florence cache
Notebook này:
- pin revision + pre-download Florence trước
- tách bước cache model và bước run detection
- load YOLO + Florence đúng **1 lần**
- xử lý nhiều `video_id` trong cùng kernel, không reload model mỗi vòng
- hỗ trợ biến HF cache thành Kaggle Dataset


In [ ]:
from pathlib import Path

# Lấy thư mục chứa notebook này
NOTEBOOK_DIR = Path().resolve()  # hoặc Path(__file__).parent nếu dùng script
REPO_ROOT = NOTEBOOK_DIR.parent.parent

In [ ]:
# !git clone https://github.com/CallmeAndree/NLP4B.git -b itshoang2024/dev
!pip -q install -r "{REPO_ROOT}/requirements.txt"
!pip -q install azure-storage-blob supabase psutil

fatal: destination path 'NLP4B' already exists and is not an empty directory.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/kaggle/working/NLP4B/data-processing/requirements.txt'


In [ ]:
import os
import sys
import shutil

import pandas as pd
from azure.storage.blob import BlobServiceClient, ContentSettings
from supabase import create_client
from dotenv import load_dotenv
load_dotenv()


# ===== Secrets =====
os.environ["AZURE_STORAGE_CONNECTION_STRING"] = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
os.environ["SUPABASE_URL"] = os.getenv("SUPABASE_URL")
os.environ["SUPABASE_KEY"] = os.getenv("SUPABASE_KEY")

# ===== Runtime config =====
RUNNER = "all"   # hoang / nanh / lam / binh
START_ID, END_ID = 0, 3

YOLO_MODEL = "yolov8m-worldv2.pt"
FLORENCE_MODEL_ID = "microsoft/Florence-2-large-ft"
FLORENCE_REVISION = "main"  # đổi sang commit hash nếu muốn pin cứng hơn
TORCH_DTYPE = "float16"     # gợi ý: float16 trên T4, float32 nếu gặp lỗi dtype
SAVE_EVERY = 1
IOU_THRESHOLD = 0.75
LIMIT_PER_VIDEO = None

# ===== HF cache config =====
HF_CACHE_ROOT = REPO_ROOT/ "data" / "hf_home"
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# Nếu đã đóng gói cache thành Kaggle Dataset thì điền path tại đây.
# Ví dụ: "/kaggle/input/florence2-cache/hf_home"
HF_DATASET_CACHE_ROOT = None

# ===== Paths =====
OBJ_DET_PATH = REPO_ROOT / "src" / "object_detection"
OUTPUT_OBJECT_DIR = Path(REPO_ROOT / "data" / "object_detection")
LOCAL_KEYFRAME_ROOT = Path(REPO_ROOT / "data" / "keyframes")
OUTPUT_OBJECT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_KEYFRAME_ROOT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(OBJ_DET_PATH))

# ===== External services =====
supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])
blob_service_client = BlobServiceClient.from_connection_string(os.environ["AZURE_STORAGE_CONNECTION_STRING"])
object_container_client = blob_service_client.get_container_client("object-detection")
try:
    object_container_client.get_container_properties()
except Exception:
    blob_service_client.create_container("object-detection")

df = pd.read_csv(REPO_ROOT / "data" / "video_id" / f"{RUNNER}_object_detection_failed.csv")
print(f"Loaded {len(df)} assigned videos for runner={RUNNER}")


c:\Users\Hoang\miniconda3\envs\agentic_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Loaded 447 assigned videos for runner=all


## Florence cache step

Chạy cell dưới **trước** khi detection:

- Nếu đã có Kaggle Dataset chứa HF cache: notebook sẽ copy cache đó về `/kaggle/working/hf_home`
- Nếu chưa có: notebook sẽ `snapshot_download()` Florence với revision đã pin
- Sau khi tải xong, bạn có thể publish thư mục cache này thành Kaggle Dataset để các lần `Save & Commit` sau không phải tải lại


In [3]:
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil
import os

def ensure_florence_cache(
    model_id: str,
    revision: str,
    cache_root: Path,
    dataset_cache_root: str | None = None,
) -> Path:
    cache_root = Path(cache_root)

    # 1) Ưu tiên dùng cache từ Kaggle Dataset
    if dataset_cache_root:
        dataset_cache_root = Path(dataset_cache_root)
        if dataset_cache_root.exists():
            print(f"[cache] Copying HF cache from Kaggle Dataset: {dataset_cache_root}")
            if cache_root.exists():
                shutil.rmtree(cache_root)
            shutil.copytree(dataset_cache_root, cache_root)
            print(f"[cache] HF cache copied to: {cache_root}")

            # Tìm thư mục model đã cache
            model_dirs = list(cache_root.glob("**/snapshots/*"))
            if model_dirs:
                local_model_dir = model_dirs[0]
                print(f"[cache] Using cached Florence dir: {local_model_dir}")
                return local_model_dir

            print("[cache] Copied dataset cache but no snapshot dir found. Will fallback to download.")
        else:
            print(f"[cache] Dataset cache path not found, fallback to download: {dataset_cache_root}")

    # 2) Nếu chưa có cache thì mới download
    print(f"[cache] snapshot_download model={model_id} revision={revision}")

    local_model_dir = snapshot_download(
        repo_id=model_id,
        revision=revision,
        cache_dir=str(cache_root),
        allow_patterns=[
            "*.json",
            "*.py",
            "*.model",
            "*.txt",
            "*.md",
            "*.safetensors",
            "*.tiktoken",
            "*.vocab",
        ],
        ignore_patterns=[
            "*.bin",
            "*.h5",
            "*.msgpack",
            "rust_model.ot",
            "tf_model.h5",
            "flax_model.msgpack",
        ],
    )

    print(f"[cache] Florence cached at: {local_model_dir}")
    return Path(local_model_dir)

LOCAL_FLORENCE_DIR = ensure_florence_cache(
    model_id=FLORENCE_MODEL_ID,
    revision=FLORENCE_REVISION,
    cache_root=HF_CACHE_ROOT,
    dataset_cache_root=HF_DATASET_CACHE_ROOT,
)
print("LOCAL_FLORENCE_DIR =", LOCAL_FLORENCE_DIR)


[cache] snapshot_download model=microsoft/Florence-2-large-ft revision=main


c:\Users\Hoang\miniconda3\envs\agentic_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 14 files: 100%|██████████| 14/14 [00:00<?, ?it/s]

[cache] Florence cached at: D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--microsoft--Florence-2-large-ft\snapshots\4a12a2b54b7016a48a22037fbd62da90cd566f2a
LOCAL_FLORENCE_DIR = D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--microsoft--Florence-2-large-ft\snapshots\4a12a2b54b7016a48a22037fbd62da90cd566f2a


## Hàm tải keyframe từ Azure


In [4]:
KEYFRAME_CONTAINER = "keyframes"

def download_video_keyframes(video_id: str, local_root: str | Path = LOCAL_KEYFRAME_ROOT) -> str:
    container_client = blob_service_client.get_container_client(KEYFRAME_CONTAINER)
    local_dir = Path(local_root) / video_id
    local_dir.mkdir(parents=True, exist_ok=True)

    blobs = list(container_client.list_blobs(name_starts_with=f"{video_id}/"))
    if not blobs:
        raise FileNotFoundError(f"No keyframes found on Azure for video_id={video_id}")

    for blob in blobs:
        filename = Path(blob.name).name
        local_path = local_dir / filename
        with open(local_path, "wb") as f:
            data = container_client.download_blob(blob.name).readall()
            f.write(data)

    return str(local_dir)


## Load models đúng 1 lần

Cell này dùng `object_detection.py` đã được revise để:
- load log rõ ràng
- dùng local HF cache
- không fetch lại remote code nếu cache đã sẵn
- tái sử dụng runtime cho toàn bộ batch


In [5]:
import object_detection as od

runtime = od.load_models(
    yolo_model_path=YOLO_MODEL,
    florence_model=str(LOCAL_FLORENCE_DIR),
    hf_cache_dir=None,                 # đã resolve cache ở cell trước
    florence_revision=FLORENCE_REVISION,
    device="cuda" if __import__("torch").cuda.is_available() else "cpu",
    torch_dtype=TORCH_DTYPE,
)

print("Runtime ready. Florence source:", runtime["florence_source"])


[object_detection] Before model load | cuda_available=False | ram_used_pct=61.9 | ram_available_gb=10.59
[object_detection] Loading YOLO model from: yolov8m-worldv2.pt
[object_detection] YOLO model loaded
[object_detection] Using local Florence path: D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--microsoft--Florence-2-large-ft\snapshots\4a12a2b54b7016a48a22037fbd62da90cd566f2a
[object_detection] Loading Florence processor from: D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--microsoft--Florence-2-large-ft\snapshots\4a12a2b54b7016a48a22037fbd62da90cd566f2a | local_files_only=True
[object_detection] Florence processor loaded
[object_detection] Loading Florence model from: D:\Code\HCMUS\NLP4B\NLP4B\data-processing\data\hf_home\models--microsoft--Florence-2-large-ft\snapshots\4a12a2b54b7016a48a22037fbd62da90cd566f2a | dtype=torch.float16 | device=cpu
[object_detection] Florence model loaded
[object_detection] After model load | cuda_available=False | ram_used

## Loop object detection cho nhiều video_id mà không reload model


In [6]:
processed_videos = []
failed_videos = []

for i in range(START_ID, min(END_ID, len(df))):
    video_id_with_ext = df.iloc[i]["video_id"]
    video_id = str(video_id_with_ext).replace(".mp4", "")
    print(f"\n[{i}] Processing video: {video_id}")

    local_input_dir = None
    combined_file = None

    try:
        # A. Download keyframes từ Azure
        local_input_dir = download_video_keyframes(video_id)

        # B. Run detection bằng runtime đã load sẵn
        combined_file = od.process_directory(
            input_dir=local_input_dir,
            output_dir=OUTPUT_OBJECT_DIR,
            runtime=runtime,
            iou=IOU_THRESHOLD,
            limit=LIMIT_PER_VIDEO,
            save_every=SAVE_EVERY,
        )

        # C. Upload JSON lên Azure
        print(f"[{i}] Uploading object detection to Azure...")
        blob_client = object_container_client.get_blob_client(f"{video_id}/{combined_file.name}")
        with open(combined_file, "rb") as data:
            blob_client.upload_blob(
                data,
                overwrite=True,
                content_settings=ContentSettings(content_type="application/json"),
            )

        # D. Update Supabase progress
        print(f"[{i}] Updating DB progress...")
        supabase.table("video_processing_progress").update({"object_detection": True}).eq("video_id", video_id).execute()

        processed_videos.append(video_id)

    except Exception as e:
        print(f"[{i}] FAILED: {video_id} | error={e}")
        failed_videos.append({"index": i, "video_id": video_id, "error": str(e)})

    finally:
        # E. Cleanup local artifacts theo từng video
        if combined_file and Path(combined_file).exists():
            Path(combined_file).unlink()

        if local_input_dir and Path(local_input_dir).exists():
            shutil.rmtree(local_input_dir)

        od.cleanup_memory()

print("\nDone.")
print("Processed videos:", processed_videos)
print("Failed videos:", failed_videos)



[0] Processing video: qOAtCbCbQmY
[object_detection] Video qOAtCbCbQmY | total_images=266 | already_processed=0 | remaining=266
[object_detection] Video qOAtCbCbQmY | [1/266] Processing qOAtCbCbQmY_00000.jpg


KeyboardInterrupt: 